# NOAA GHCN Weather Data Pipeline for Africa
### A Practical Guide to Station Discovery, Data Ingestion, and Climate Analysis

---

### Overview
NOAA's Global Historical Climatology Network (GHCN) provides decades of global surface weather data. However, working with it directly is challenging because data is split across raw S3 Parquet buckets, integer tenths scaling (e.g., $254$ for $25.4$ mm), missing value sentinels (`-9999`), and fixed-width text inventories.

`masharikiweather` provides a clean Python wrapper (`ghcn_apiwrapper.py`) that handles all of this automatically:
- **Discovers stations** by region, country name, or bounding box coordinates.
- **Inspects sensor lifecycles** independently (so an ended thermometer doesn't crash an active rain query).
- **Downloads and cleans data** concurrently with unit normalization (tenths $\rightarrow$ mm and °C).
- **Safely extracts time series** without `KeyError` crashes.
- **Visualizes data completeness** and missing days using heatmaps.
- **Aligns ground stations** with ERA5 gridded reanalysis.

| Key Feature | Function / Method | What it does |
| :--- | :--- | :--- |
| **Station Discovery** | `GHCNDaily.get_african_stations(...)` | Finds stations by region, country, or coordinate box. |
| **Sensor Activeness** | `GHCNDaily.get_stations_variable_activeness(...)` | Checks independent start/end years for each sensor. |
| **Daily Ingestion** | `query_ghcn_daily(...)` | One-line multi-station, multi-variable clean download. |
| **Safe Extraction** | `get_station_series(df, station, variable)` | Extracts a station's series safely without `KeyError`. |
| **Hourly Ingestion** | `query_ghcn_hourly(...)` | Sub-daily telemetry from airport AWOS/ASOS stations. |
| **Grid Alignment** | `pipeline.align(target_meta, ...)` | Pairs ground stations to ERA5 0.25° grid centroids. |


## 1. Environment Setup & Imports

We start by loading our libraries and importing the pipeline tools from `masharikiweather`.

In [ ]:
# Query airport hourly observations (JKIA Nairobi and Wilson)
hourly_df = query_ghcn_hourly(
    stations=["HKJK", "HKNW"],
    variables=["t2m", "d2m", "rh", "wind_speed"],
    start_date="2024-07-01",
    end_date="2024-07-05",
    format="long",
    access_method="auto"  # Options: 'auto', 'by_year', or 'by_station'
)

if not hourly_df.empty:
    print(f"Retrieved {len(hourly_df)} hourly observations:")
    display(hourly_df.head(6))
    
    # Inspect summary statistics for JKIA and Wilson
    print("\nMean temperature and humidity by station:")
    display(hourly_df.groupby(["station", "variable"])["observation"].mean().unstack())
else:
    print("Hourly observations not present in local cache for test window.")


## 2. Station Discovery Methods

Before downloading data, we need to know which stations exist and where they are located.  
`GHCNDaily.get_african_stations(...)` provides three ways to discover stations:
1. **By Region Preset**: Built-in African geographic presets (`'kenya'`, `'east_africa'`, `'west_africa'`, `'southern_africa'`, `'central_africa'`, `'north_africa'`, `'africa'`).
2. **By Country Name**: Any country or list of countries (e.g. `'Nigeria'`, `['Kenya', 'Tanzania']`).
3. **By Geographic Bounding Box**: Coordinate limits `(min_lat, min_lon, max_lat, max_lon)` (e.g. Mali / Sahel: `(10.0, -12.25, 25.0, 4.25)`).

Let us test all three methods:

In [ ]:
# --- Method 1: Discover Active Stations in Kenya (2020 to Date) ---
ke_stations = GHCNDaily.get_african_stations(
    region="kenya",
    active_only=True,
    min_year=2020,
    include_inventory=True
)
print(f"Method 1 (Region 'kenya'): Found {len(ke_stations)} active stations.")
display(ke_stations[['ID', 'NAME', 'LAT', 'LON', 'ELEV', 'status', 'elements_available']].head(6))

# --- Method 2: Discover Stations in Nigeria and Ghana ---
west_stations = GHCNDaily.get_african_stations(
    countries=["Nigeria", "Ghana"],
    include_inventory=True
)
print(f"\nMethod 2 (Countries 'Nigeria' & 'Ghana'): Found {len(west_stations)} stations.")
display(west_stations[['ID', 'NAME', 'country_name', 'status', 'elements_available']].head(6))

# --- Method 3: Discover Stations in a Custom Bounding Box (Mali: 10°N to 25°N, 12.25°W to 4.25°E) ---
bbox_stations = GHCNDaily.get_african_stations(
    bounds=(10.0, -12.25, 25.0, 4.25), # (min_lat, min_lon, max_lat, max_lon)
    include_inventory=True
)
print(f"\nMethod 3 (Mali Bounding Box): Found {len(bbox_stations)} stations.")
display(bbox_stations[['ID', 'NAME', 'LAT', 'LON', 'status']].head(6))

### Interactive Station Map
To inspect station coverage visually, we can plot them on an interactive `folium` map.  
- **Green pins** = Active stations reporting in recent years.
- **Crimson pins** = Historical stations that ceased operations.

In [ ]:
# Plot Kenya's active stations on an interactive map
if not ke_stations.empty:
    center_lat = ke_stations['LAT'].astype(float).mean()
    center_lon = ke_stations['LON'].astype(float).mean()
    station_map = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles='OpenStreetMap')

    for _, row in ke_stations.iterrows():
        status = str(row.get('status', 'Unknown'))
        color = 'forestgreen' if 'Active' in status else 'crimson'
        popup_text = (
            f"<b>Station:</b> {row['ID']}<br>"
            f"<b>Name:</b> {row['NAME']}<br>"
            f"<b>Status:</b> {status}<br>"
            f"<b>Elements:</b> {row.get('elements_available', 'N/A')}<br>"
            f"<b>Elevation:</b> {row.get('ELEV', 'N/A')} m"
        )
        folium.CircleMarker(
            location=[float(row['LAT']), float(row['LON'])],
            radius=7,
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"{row['ID']} - {row['NAME']}",
            color=color,
            fill=True,
            fillOpacity=0.75
        ).add_to(station_map)

    display(station_map)

## 3. Sensor Activeness & Independent Lifecycles

A common pitfall in African climatology is assuming all sensors at a station start and end at the same time. In reality, a station's manual rain gauge might be active today, but its automatic thermometer broke in 2015.  

`GHCNDaily.get_stations_variable_activeness(...)` checks the lifecycle of each sensor independently so we know exactly what is available before requesting data.

In [ ]:
# Check independent sensor lifecycles for sample stations
sample_ids = ke_stations['ID'].head(5).tolist()
sensor_lifecycles = GHCNDaily.get_stations_variable_activeness(stations=sample_ids)

display(sensor_lifecycles[['station', 'name', 'rain_status', 'tmax_status', 'wind_status', 'active_variables', 'ended_variables']])

## 4. Fetching Daily Weather Observations

`query_ghcn_daily(...)` downloads, quality-checks, and scales observations in a single call:
- Raw integer tenths are divided by 10 (tenths of mm $\rightarrow$ mm, tenths of °C $\rightarrow$ °C).
- Sentinel values (`-9999`) and impossible readings are removed.
- Diurnal Temperature Range ($DTR = T_{\max} - T_{\min}$) is computed automatically if requested.
- Formats the result as a Wide matrix (`Date` $\times$ `Stations/Variables`).

In [ ]:
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

print(f"Fetching daily observations across Kenya for {START_DATE[:4]}...")
daily_wide_df = query_ghcn_daily(
    region="kenya",
    variables=["tp", "t2m_max", "t2m_min", "dtr"],
    start_date=START_DATE,
    end_date=END_DATE,
    format="wide",
    active_only=True,
    min_year=2020
)

print(f"Shape: {daily_wide_df.shape} (Days, Station-Variable combinations)")
display(daily_wide_df.head())

## 5. Missing Data & Network Coverage Heatmap

Weather telemetry often contains gaps due to sensor maintenance or transmission delays.  
A missing data heatmap gives us an instant view of data completeness across all stations over the year:  
- **Purple** = Valid recorded observation.
- **Yellow** = Missing observation (`NaN`).

In [ ]:
# Extract precipitation ('tp') across all stations
if not daily_wide_df.empty and isinstance(daily_wide_df.columns, pd.MultiIndex):
    prcp_matrix = daily_wide_df.xs('tp', axis=1, level=1)

    plt.figure(figsize=(14, 5))
    sns.heatmap(prcp_matrix.isna(), cbar=False, cmap='viridis', yticklabels=False)
    plt.title(f"Precipitation Data Gaps Across Kenya in {START_DATE[:4]} (Yellow = Missing, Purple = Observed)", fontsize=13)
    plt.xlabel("Station ID", fontsize=11)
    plt.ylabel(f"Time ({START_DATE} to {END_DATE})", fontsize=11)
    plt.tight_layout()
    plt.show()

## 6. Safe Time Series Extraction & Station Analytics

When analyzing a specific station, using `df[station][variable]` will crash with a `KeyError` if that station didn't record that variable.  

`get_station_series(df, station, variable)` extracts a 1D `pd.Series` safely without crashing, returning an empty Series if missing.  

Let us examine **Eldoret International Airport (`KEM00063686`)** across 3 synchronized panels:
1. **Precipitation (mm)**: Daily rainfall bars and seasonal total.
2. **Temperature Extremes (°C)**: Maximum and Minimum temperatures.
3. **Diurnal Temperature Range (DTR in °C)**: Day-night temperature difference ($T_{\max} - T_{\min}$).

In [ ]:
TARGET_STATION = "KEM00063686" # Eldoret Intl
st_name = ke_stations.loc[ke_stations['ID'] == TARGET_STATION, 'NAME'].values[0] if TARGET_STATION in ke_stations['ID'].values else TARGET_STATION

# Safely extract series (guaranteed no KeyError)
s_rain = get_station_series(daily_wide_df, station=TARGET_STATION, variable="tp")
s_tmax = get_station_series(daily_wide_df, station=TARGET_STATION, variable="t2m_max")
s_tmin = get_station_series(daily_wide_df, station=TARGET_STATION, variable="t2m_min")
s_dtr  = get_station_series(daily_wide_df, station=TARGET_STATION, variable="dtr")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 1. Daily Rainfall
if not s_rain.empty and s_rain.dropna().count() > 0:
    clean_rain = s_rain.dropna()
    axes[0].bar(clean_rain.index, clean_rain.values, color='teal', width=1.0, alpha=0.8)
    axes[0].set_title(f"Daily Precipitation for {TARGET_STATION} ({st_name}) - Total: {clean_rain.sum():.1f} mm", fontsize=12)
    axes[0].set_ylabel("Rainfall (mm)")
    axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Temperature Extremes
if not s_tmax.empty and s_tmax.dropna().count() > 0:
    axes[1].plot(s_tmax.index, s_tmax.values, color='crimson', label='Tmax (°C)', lw=1.5)
    if not s_tmin.empty and s_tmin.dropna().count() > 0:
        axes[1].plot(s_tmin.index, s_tmin.values, color='royalblue', label='Tmin (°C)', lw=1.5)
    axes[1].set_title(f"Temperature Extremes for {TARGET_STATION} ({st_name})", fontsize=12)
    axes[1].set_ylabel("Temperature (°C)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. Diurnal Temperature Range (DTR)
if not s_dtr.empty and s_dtr.dropna().count() > 0:
    axes[2].plot(s_dtr.index, s_dtr.values, color='purple', label='DTR (Tmax - Tmin)', lw=1.5)
    axes[2].set_title(f"Diurnal Temperature Range (DTR) - Mean: {s_dtr.dropna().mean():.1f} °C", fontsize=12)
    axes[2].set_ylabel("DTR (°C)")
    axes[2].set_xlabel("Date")
    axes[2].legend(loc="upper right")
    axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Long Format for Machine Learning

For training machine learning models or Graph Neural Networks (GNNs), a long tabular format is preferred over a sparse matrix.  
Specifying `format='long'` yields a standardized table with columns: `['time', 'lat', 'lon', 'station', 'observation', 'variable']`.

In [ ]:
pipeline = GHCN(resolution="daily", region="kenya", active_only=True, min_year=2020)
daily_long_df = pipeline(
    variables=["tp", "t2m_max", "t2m_min"],
    start_date=START_DATE,
    end_date=END_DATE,
    format="long"
)

print(f"Long Table Shape: {daily_long_df.shape}")
display(daily_long_df.head())

print("\nSummary statistics across variables:")
display(daily_long_df.groupby("variable")["observation"].describe())

## 8. Cross-Country African Exploration

The pipeline works seamlessly across all African countries. Let us query an active Southern African synoptic station: **Maun, Botswana (`BC000068032`)** to inspect its semi-arid climate telemetry.

In [ ]:
BW_STATION = "BC000068032" # Maun, Botswana

print(f"Fetching climate data for Maun, Botswana ({BW_STATION}) for 2024...")
bw_df = query_ghcn_daily(
    stations=[BW_STATION],
    variables=["rain", "tmax", "tmin", "dtr"],
    start_date="2024-01-01",
    end_date="2024-12-31",
    format="wide"
)

display(bw_df.head())

bw_rain = get_station_series(bw_df, station=BW_STATION, variable="rain")
bw_tmax = get_station_series(bw_df, station=BW_STATION, variable="tmax")
bw_dtr  = get_station_series(bw_df, station=BW_STATION, variable="dtr")

print(f"\nClimate Analytics for Maun, Botswana (2024):")
print(f"  • Total Rainfall:       {bw_rain.dropna().sum():.1f} mm across {bw_rain.dropna().count()} rain days")
print(f"  • Mean Max Temperature: {bw_tmax.dropna().mean():.1f} °C")
print(f"  • Mean Diurnal Range:   {bw_dtr.dropna().mean():.1f} °C")

## 9. Sub-Daily Telemetry (GHCN-Hourly)

For convective storm tracking and diurnal wind studies, airport stations report sub-daily hourly telemetry (temperature `t2m`, dew point `d2m`, relative humidity `rh`, wind speed `wind_speed`).

NOAA NCEI hosts hourly data on their object store with two primary access methods:
1. **`by_year` (Default/Auto)**: Annual slices (`.parquet` or `.psv`) for specific query years (~500 KB/station-year).
2. **`by_station`**: Full Period-of-Record (POR) archives (`.psv`) from [`hourly/access/by-station/`](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/index.html#hourly/access/by-station/).

The wrapper automatically resolves station identifiers (e.g. airport ICAO codes like `HKJK` and `HKNW`, or WMO IDs) to their official NOAA GHCNh IDs (`KEI0000HKJK`, `KEI0000HKNW`), applies physical QC, and caches data locally.

In [ ]:
# Query airport hourly observations (JKIA Nairobi and Wilson)
hourly_df = query_ghcn_hourly(
    stations=["HKJK", "HKNW"],
    variables=["t2m", "d2m", "rh", "wind_speed"],
    start_date="2024-07-01",
    end_date="2024-07-05",
    format="long",
    access_method="auto"  # Options: 'auto', 'by_year', or 'by_station'
)

if not hourly_df.empty:
    print(f"Retrieved {len(hourly_df)} hourly observations:")
    display(hourly_df.head(6))
    
    # Inspect summary statistics for JKIA and Wilson
    print("\nMean temperature and humidity by station:")
    display(hourly_df.groupby(["station", "variable"])["observation"].mean().unstack())
else:
    print("Hourly observations not present in local cache for test window.")


## 10. Spatial Alignment with ERA5 Reanalysis Grid

To evaluate or bias-correct climate models, `pipeline.align(...)` calculates great-circle Haversine distances to match each ground station to its closest 0.25° ERA5 reanalysis grid point.

In [ ]:
# Construct 0.25° ERA5 coordinate centroids over Kenya
grid_lats = np.arange(-4.75, 5.51, 0.25)
grid_lons = np.arange(33.75, 42.01, 0.25)
mesh_lat, mesh_lon = np.meshgrid(grid_lats, grid_lons)

era5_meta = pd.DataFrame({
    "station_id": [f"ERA5_{lat:.2f}_{lon:.2f}" for lat, lon in zip(mesh_lat.ravel(), mesh_lon.ravel())],
    "latitude": mesh_lat.ravel(),
    "longitude": mesh_lon.ravel(),
})

# Find closest grid centroid for each GHCN station
era5_alignment = pipeline.align(target_meta=era5_meta, target_name="era5")
display(era5_alignment.head(6))

## 11. Exporting Clean Data

Finally, export the cleaned datasets and alignment tables to disk in fast, portable Parquet and CSV formats.

In [ ]:
output_dir = os.path.abspath("processed_ghcn_output")
os.makedirs(output_dir, exist_ok=True)

wide_path = os.path.join(output_dir, f"ghcn_kenya_wide_{START_DATE[:4]}.parquet")
long_path = os.path.join(output_dir, f"ghcn_kenya_long_{START_DATE[:4]}.parquet")
align_path = os.path.join(output_dir, "ghcn_kenya_era5_alignment.csv")

daily_wide_df.to_parquet(wide_path)
daily_long_df.to_parquet(long_path, index=False)
era5_alignment.to_csv(align_path, index=False)

print("✓ Successfully exported:")
print(f"  • Wide observations: {wide_path}")
print(f"  • Long format:       {long_path}")
print(f"  • Grid alignment:    {align_path}")